# 3. GPT Realtime Integration

**Agenda slot: 35–110 min**

The Unified Realtime Model from notebook 2: one persistent session instead of
chained STT/LLM/TTS services.

```
Browser
 ↓ WebRTC/WebSocket
GPT Realtime API
 ↓
Audio In → Reasoning → Audio Out
```

Key pieces:

- **Session** — one long-lived connection for the whole conversation
- **Events** — every state change is a typed event on that connection
- **Audio chunks** — streamed both directions, not sent as whole files
- **Conversation state** — held server-side for the life of the session
- **Streaming responses** — text/audio arrive incrementally, not all at once

Full agenda: [`../voice-agent.md`](../voice-agent.md)

## What we're building

```
Browser → Mic → Realtime API → Function Calling → FastAPI
                                                      ↓
                                   get_weather / calculate / get_order_status
                                                      ↓
                                        Realtime → Speaker
```

This is the exact shape of the app in `../backend`.

## → Live Demo Break

The mic and speaker in a real Realtime session go through the browser's
WebRTC stack, not this notebook — so start the actual app now, in a terminal
from the repo root:

```bash
source .venv/bin/activate
uvicorn backend.main:app --reload
```

Open **http://localhost:8000**, click **Start talking**, and walk through:

1. Create session
2. Connect browser (WebRTC)
3. Capture microphone
4. Send audio
5. Receive audio
6. Play response

Keep DevTools open on the Network/Console tab — the next section is easier to
follow once people have seen the raw events fly by.

Once the server is running, the cells below call it directly from Python —
the same requests the browser makes — so we can inspect the session and tool
responses without needing DevTools.

In [ ]:
import requests

session = requests.post("http://localhost:8000/realtime/session").json()
list(session.keys())

That's the ephemeral client secret the browser uses to open its own WebRTC
connection straight to OpenAI — our backend never sees the audio, it only
mints this short-lived token (see `backend/api/realtime.py`).

## Events Deep Dive

Every state change on the session is one of these events. Point them out live
in DevTools as the conversation runs — students love seeing the raw stream.

In [ ]:
["session.created", "input_audio_buffer.append", "input_audio_buffer.commit", "response.create", "response.audio.delta", "response.audio.done", "response.done"]

## Tool Calling

```
User: "I want to order pizza"
 ↓
GPT → Function Call → FastAPI → JSON → GPT → Voice Response
```

The model never sees SQL, REST endpoints, or internal APIs — only a tool name
and a JSON schema for its arguments. Ours:

- `get_weather(location)`
- `calculate(expression)`
- `get_order_status(order_id)`

Every one of these is dispatched through a single `POST /tools/call` endpoint
on the backend (see `backend/api/tools.py`). With the server running, we can
call it directly, exactly like the browser does after a function call:

In [ ]:
requests.post("http://localhost:8000/tools/call", json={"name": "get_weather", "arguments": {"location": "Pune"}}).json()

## Prompt Engineering for Voice

Chat prompts and voice prompts are not the same thing.

**Instead of:**
```
You are helpful.
```

**Use:**
```
Be concise. Speak naturally.
Never exceed 2 sentences.
Avoid markdown.
Pause before answering.
Confirm actions.
Ask one question at a time.
```

Why: voice output is *heard*, not skimmed. Markdown, long paragraphs, and
bullet lists all sound wrong out loud — see `backend/core/agent-instruction.md`
for the prompt actually driving today's demo.

## Advanced Features (mention only)

- Voice selection
- Interruptions (barge-in)
- Voice Activity Detection (VAD)
- Streaming transcripts
- Emotion
- Background noise handling
- Multi-language support
- Context window management

## Q&A + Next Steps

Where this goes from here:

- MCP integration
- Calendar booking
- RAG
- CRM integration
- Twilio phone agent
- SIP calling
- Browser agent
- Customer support agent

### Closing live demo flow

Back in the browser, run through one exciting end-to-end conversation:

```
"Hi" → AI responds
"What's the weather in Pune?" → calls get_weather → answers
"What's 18 times 24?" → calls calculate → answers
"What's the status of order ORD-1023?" → calls get_order_status → answers
```

One flow, three tool calls, zero typing.